### Import libraries

In [19]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

pd.set_option("display.max_colwidth", 150)

sns.set_theme(style="whitegrid")

### Set project paths

In [20]:
ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

TRAIN_PATH = ROOT / "data" / "processed" / "train.csv"
OUTPUT_DIR = ROOT / "data" / "processed"

df = pd.read_csv(TRAIN_PATH)

### Validate required columns

In [21]:
TEXT_COLUMN = "clean_text"
SENTIMENT_COLUMN = "sentiment"

required_columns = [
    TEXT_COLUMN,
    SENTIMENT_COLUMN
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}"
    )

### Basic whitespace tokenization

In [22]:
text = str(df[TEXT_COLUMN].iloc[0])
tokens = text.split()

print("Original text:")
print(text)

Original text:
great clarity and sound.. volume is above my previous unit which was a jawbone.. recommend for all.. battery life only issue with motorolla


In [23]:
print(tokens)

['great', 'clarity', 'and', 'sound..', 'volume', 'is', 'above', 'my', 'previous', 'unit', 'which', 'was', 'a', 'jawbone..', 'recommend', 'for', 'all..', 'battery', 'life', 'only', 'issue', 'with', 'motorolla']


In [24]:
print(len(tokens))

23


### Create a reusable tokenization function

In [25]:
def whitespace_tokenize(text):
    """
    Split text into tokens using whitespace.
    """
    if pd.isna(text):
        return []

    return str(text).split()


sample_text = "This mobile phone has an excellent camera."

print(whitespace_tokenize(sample_text))

['This', 'mobile', 'phone', 'has', 'an', 'excellent', 'camera.']


### Word tokenization using regular expressions

In [26]:
def regex_tokenize(text):
    """
    Tokenize words, contractions, and punctuation.
    """
    if pd.isna(text):
        return []

    text = str(text).lower()

    pattern = r"\w+(?:['’]\w+)*|[^\w\s]"

    return re.findall(pattern, text)


sample_text = "I don't like this phone! The camera is good."

print("Original text:")
print(sample_text)

print("\nRegex tokens:")
print(regex_tokenize(sample_text))

Original text:
I don't like this phone! The camera is good.

Regex tokens:
['i', "don't", 'like', 'this', 'phone', '!', 'the', 'camera', 'is', 'good', '.']


### Tokenize the entire dataset

In [29]:
df["tokens"] = df[TEXT_COLUMN].apply(whitespace_tokenize)

df["token_count"] = df["tokens"].apply(len)

print("Tokenization completed!")

display(
    df[[TEXT_COLUMN, SENTIMENT_COLUMN, "tokens", "token_count"]]
    .head(3)
)

Tokenization completed!


,clean_text,sentiment,tokens,token_count
0,great clarity and sound.. volume is above my previous unit which was a jawbone.. recommend for all.. battery life only issue with motorolla,positive,"[great, clarity, and, sound.., volume, is, above, my, previous, unit, which, was, a, jawbone.., recommend, for, all.., battery, life, only, issue,...",23
1,i do not like the way it fits in my ear. i have tried all the ear tips but none of them fit me well. it is very loose and i use it mainly at home ...,negative,"[i, do, not, like, the, way, it, fits, in, my, ear., i, have, tried, all, the, ear, tips, but, none, of, them, fit, me, well., it, is, very, loose...",103
2,"fits well, syncs up with phones very easily and has good range and sound quality. definitely worth it for the price.",positive,"[fits, well,, syncs, up, with, phones, very, easily, and, has, good, range, and, sound, quality., definitely, worth, it, for, the, price.]",21


### Validate tokenization results

In [31]:
assert "tokens" in df.columns
assert "token_count" in df.columns

assert df["tokens"].apply(
    lambda value: isinstance(value, list)
).all()

assert df["token_count"].equals(
    df["tokens"].apply(len)
)

print("Tokenization validation passed!")
print("All reviews have valid token lists.")

Tokenization validation passed!
All reviews have valid token lists.


### Save the tokenized dataset

In [32]:
OUTPUT_PATH = OUTPUT_DIR / "train_tokenized.csv"

output_df = df.copy()

output_df["tokens"] = output_df["tokens"].apply(
    lambda tokens: " ".join(tokens)
)

output_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)